# Data Exploration

A ideia aqui é apenas inspecionar os dados, confirmar a chave de cruzamento, executar um `join` simples e validar o resultado.

In [1]:
import sys

print("Python:", sys.version)
print("Executable:", sys.executable)

Python: 3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)]
Executable: C:\Users\beelt\Documents\collections_case_candidate\.venv\Scripts\python.exe


In [2]:
import pandas as pd

pd.set_option('display.max_columns', None)

## 1. Carregando os dados

In [3]:
dataset1 = pd.read_csv('../data/raw/collections_queue_sep2026.csv')
dataset2 = pd.read_csv('../data/raw/whatsapp_collections_history.csv')

## 2. Shape das bases

In [4]:
print('collections_queue:', dataset1.shape)
print('whatsapp:', dataset2.shape)

collections_queue: (10658, 10)
whatsapp: (75406, 17)


## 3. Visualização rápida

In [5]:
dataset1.head()

,customer_id,in_collections_since,days_past_due_on_2026-09-01,outstanding_balance_brl,monthly_salary_brl,payday_day_of_month,n_prior_transactions,account_age_months,days_since_last_app_login,state_uf
0,C000002,2026-07-26,38,1143.79,2670.0,20,4,7,49,BA
1,C000005,2026-08-05,28,250.35,3200.0,5,1,5,46,RS
2,C000011,2026-07-13,51,509.53,3320.0,1,19,17,15,RS
3,C000014,2026-08-02,31,1032.03,2310.0,20,10,11,10,SP
4,C000015,2026-07-22,42,1458.43,2580.0,30,8,3,33,GO


In [15]:
column_info = pd.DataFrame({
    'column': dataset1.columns,
    'dtype': dataset1.dtypes.astype(str).values
})

print('qtt cols:', len(column_info))
display(column_info)

qtt cols: 10

columns and types:
customer_id: str
in_collections_since: str
days_past_due_on_2026-09-01: int64
outstanding_balance_brl: float64
monthly_salary_brl: float64
payday_day_of_month: int64
n_prior_transactions: int64
account_age_months: int64
days_since_last_app_login: int64
state_uf: str


In [6]:
dataset2.head()

,message_id,customer_id,sent_at,template,n_msgs_last_14d,days_past_due,outstanding_balance_brl,monthly_salary_brl,payday_day_of_month,n_prior_transactions,account_age_months,days_since_last_app_login,state_uf,delivery_status,interaction,paid_within_72h,amount_paid_brl
0,M0000001,C001934,2026-06-01 09:18,pix_link,0,1,867.63,3000.0,5,2,4,44,PB,delivered,none,0,0.0
1,M0000002,C007836,2026-06-01 09:18,friendly_reminder,0,1,790.21,3260.0,30,3,3,9,RJ,delivered,read,0,0.0
2,M0000003,C004887,2026-06-01 09:48,friendly_reminder,0,1,1030.00,3120.0,5,2,6,5,PA,delivered,none,0,0.0
3,M0000004,C009207,2026-06-01 09:53,friendly_reminder,0,1,250.94,1200.0,30,11,11,26,SP,delivered,read,0,0.0
4,M0000005,C003478,2026-06-01 09:54,friendly_reminder,0,1,511.52,1740.0,10,2,1,23,SP,delivered,read,0,0.0


In [17]:
column_info = pd.DataFrame({
    'column': dataset2.columns,
    'dtype': dataset2.dtypes.astype(str).values
})

print('qtt cols:', len(column_info))
display(column_info)

qtt cols: 17


,column,dtype
0,message_id,str
1,customer_id,str
2,sent_at,str
3,template,str
4,n_msgs_last_14d,int64
5,days_past_due,int64
6,outstanding_balance_brl,float64
7,monthly_salary_brl,float64
8,payday_day_of_month,int64
9,n_prior_transactions,int64


## 4. Identificando a chave de cruzamento

In [10]:
common_columns = sorted(set(dataset1.columns) & set(dataset2.columns))
common_columns

['account_age_months',
 'customer_id',
 'days_since_last_app_login',
 'monthly_salary_brl',
 'n_prior_transactions',
 'outstanding_balance_brl',
 'payday_day_of_month',
 'state_uf']

## 5. Conferindo a granularidade

In [9]:
print('CUSTOMER DATASET')
print('customer - rows:', len(dataset1))
print('customer - customer unique:', dataset1['customer_id'].nunique())
print('customer - duplicates in customer_id:', dataset1['customer_id'].duplicated().sum())
print('customer - missing customer_id:', dataset1['customer_id'].isna().sum())

print()

print('WHATSAPP DATASET')
print('whatsapp - rows:', len(dataset2))
print('whatsapp - unique send attempts:', dataset2['message_id'].nunique())
print('whatsapp - duplicates in message_id:', dataset2['message_id'].duplicated().sum())
print('whatsapp - missing message_id:', dataset2['message_id'].isna().sum())
print('whatsapp - customer unique:', dataset2['customer_id'].nunique())
print('whatsapp - missing customer_id:', dataset2['customer_id'].isna().sum())

# Referential integrity
customer_ids = set(dataset1['customer_id'].dropna())
whatsapp_customer_ids = set(dataset2['customer_id'].dropna())

whatsapp_wno_customer = whatsapp_customer_ids - customer_ids

print()
print('REFERENTIAL INTEGRITY')
print('whatsapp - customers not found in customer dataset:', len(whatsapp_wno_customer))
print(
    'whatsapp - rows from customers not found in customer dataset:',
    dataset2['customer_id'].isin(whatsapp_wno_customer).sum()
)

CUSTOMER DATASET
customer - rows: 10658
customer - customer unique: 10658
customer - duplicates in customer_id: 0
customer - missing customer_id: 0

WHATSAPP DATASET
whatsapp - rows: 75406
whatsapp - unique send attempts: 75406
whatsapp - duplicates in message_id: 0
whatsapp - missing message_id: 0
whatsapp - customer unique: 11724
whatsapp - missing customer_id: 0

REFERENTIAL INTEGRITY
whatsapp - customers not found in customer dataset: 6342
whatsapp - rows from customers not found in customer dataset: 42059


Interpretação esperada:

- `customer`: one row per `customer`;
- `whatsapp`: one row per `message_id`;
- Relationship between datasets: `customer 1:N whatsapp`.

## 6. Cobertura da chave

In [11]:
customer_wno_whatsapp = set(dataset1['customer_id']) - set(dataset2['customer_id'])
whatsapp_wno_customer = set(dataset2['customer_id']) - set(dataset1['customer_id'])

print('clientes w/no whatsapp:', len(customer_wno_whatsapp))
print('whatsapp w/no customer :', len(whatsapp_wno_customer))

clientes w/no whatsapp: 5276
whatsapp w/no customer : 6342


## 7. Join simples

In [13]:
dataset_integrated = dataset1.merge(
    dataset2,
    on='customer_id',
    how='left',
    validate='one_to_many'
)

dataset_integrated.shape

(38623, 26)

Como a relação é `1:N`, é esperado que a quantidade de linhas aumente após o join.

## 8. Validação rápida do resultado

In [14]:
print('shape:', dataset_integrated.shape)
print('customer:', dataset_integrated['customer_id'].nunique())
print('whatsapp unique:', dataset_integrated['message_id'].nunique())
print('whatsapp duplicates:', dataset_integrated['message_id'].duplicated().sum())

shape: (38623, 26)
customer: 10658
whatsapp unique: 33347
whatsapp duplicates: 5275


In [21]:
dataset_integrated.head(10)

,customer_id,in_collections_since,days_past_due_on_2026-09-01,outstanding_balance_brl_x,monthly_salary_brl_x,payday_day_of_month_x,n_prior_transactions_x,account_age_months_x,days_since_last_app_login_x,state_uf_x,message_id,sent_at,template,n_msgs_last_14d,days_past_due,outstanding_balance_brl_y,monthly_salary_brl_y,payday_day_of_month_y,n_prior_transactions_y,account_age_months_y,days_since_last_app_login_y,state_uf_y,delivery_status,interaction,paid_within_72h,amount_paid_brl
0,C000002,2026-07-26,38,1143.79,2670.0,20,4,7,49,BA,M0040569,2026-07-29 10:31,pix_link,0.0,4.0,1143.79,2670.0,20.0,4.0,7.0,15.0,BA,delivered,read,0.0,0.0
1,C000002,2026-07-26,38,1143.79,2670.0,20,4,7,49,BA,M0041668,2026-07-30 09:31,friendly_reminder,1.0,5.0,1143.79,2670.0,20.0,4.0,7.0,16.0,BA,delivered,none,0.0,0.0
2,C000002,2026-07-26,38,1143.79,2670.0,20,4,7,49,BA,M0044923,2026-08-02 10:54,urgent_reminder,2.0,8.0,1143.79,2670.0,20.0,4.0,7.0,19.0,BA,delivered,read,0.0,0.0
3,C000002,2026-07-26,38,1143.79,2670.0,20,4,7,49,BA,M0053395,2026-08-10 14:53,pix_link,3.0,16.0,1143.79,2670.0,20.0,4.0,7.0,27.0,BA,delivered,none,0.0,0.0
4,C000005,2026-08-05,28,250.35,3200.0,5,1,5,46,RS,M0048564,2026-08-05 14:36,friendly_reminder,0.0,1.0,250.35,3200.0,5.0,1.0,5.0,19.0,RS,delivered,read,0.0,0.0
5,C000005,2026-08-05,28,250.35,3200.0,5,1,5,46,RS,M0052190,2026-08-09 09:07,friendly_reminder,1.0,5.0,250.35,3200.0,5.0,1.0,5.0,23.0,RS,delivered,none,0.0,0.0
6,C000005,2026-08-05,28,250.35,3200.0,5,1,5,46,RS,M0054595,2026-08-11 14:41,pix_link,2.0,7.0,250.35,3200.0,5.0,1.0,5.0,25.0,RS,delivered,none,0.0,0.0
7,C000005,2026-08-05,28,250.35,3200.0,5,1,5,46,RS,M0060068,2026-08-17 11:14,urgent_reminder,3.0,13.0,250.35,3200.0,5.0,1.0,5.0,31.0,RS,delivered,none,0.0,0.0
8,C000005,2026-08-05,28,250.35,3200.0,5,1,5,46,RS,M0069365,2026-08-25 19:13,urgent_reminder,2.0,21.0,250.35,3200.0,5.0,1.0,5.0,39.0,RS,delivered,none,0.0,0.0
9,C000011,2026-07-13,51,509.53,3320.0,1,19,17,15,RS,M0024673,2026-07-13 11:20,friendly_reminder,0.0,1.0,966.15,3320.0,1.0,19.0,17.0,5.0,RS,delivered,none,0.0,0.0
